In [2]:
#import modules and load the data
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import cross_val_score
from numpy import mean
from numpy import absolute
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GridSearchCV
import tensorflow as tf
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import KFold
import optuna

df = pd.read_csv("insurance.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [3]:
#Prepare the data
df = df.dropna() #remove missings
y = df['charges'] #select Target
X = df.drop('charges',axis=1) #Select features
X = pd.get_dummies(X, drop_first=True) #Turn Fuel_Type into dummy variables
columns = X.columns #create index with column names (needed for last step)
scaler = MinMaxScaler() #initiate the scaler
X = scaler.fit_transform(X) #scale the data
X = pd.DataFrame(X,columns=columns) #turn back into a dataframe

In [4]:
#Exercise 1

#Neural network with cross validation based on the hypertuned results

#First define a function that contains the neural network model
def create_baseline():
    # create model
    nnmodel = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(X.shape[1],)),  # Input layer 
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(256, activation='relu'), #hidden layer
        tf.keras.layers.Dense(1)  # Output layer
    ])
    # Compile model
    nnmodel.compile(loss='mean_absolute_error', #loss is calculated with MAE
                    optimizer='adam', #type of gradient descent
                    metrics=['mae']) #use MAE to evaluate the model
    return nnmodel

# evaluate model 

#define the estimator
estimator = KerasRegressor(build_fn=create_baseline, #you have to use the kerasRegressor wrapper for nn models
                           epochs=200, batch_size=64, #100 epochs and 5 batches
                           verbose=0) #this will shorten the output text
#Define the kfold function
kfold = RepeatedKFold(n_splits=5, n_repeats=1, random_state=42) #I'm using just 5 splits and 1 repeat to save time, but more splits and repeats will be more accurate
#Generate the results
results = cross_val_score(estimator, X, y, cv=kfold,scoring='neg_mean_absolute_error')
#Print the results
print('the average prediction error is: %.0f' % mean(absolute(results)))

c:\Users\Tijmen\anaconda3\envs\env2526\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\Tijmen\anaconda3\envs\env2526\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\Tijmen\anaconda3\envs\env2526\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


c:\Users\Tijmen\anaconda3\envs\env2526\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\Tijmen\anaconda3\envs\env2526\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)


the average prediction error is: 1592


In [9]:
#Exercise 2
# Neural network trained on the entire dataset (no train/test split, no cross-validation)

def create_baseline():
    # create model
    nnmodel = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(X.shape[1],)),  # Input layer
        tf.keras.layers.Dense(256, activation='relu'),  # hidden layer
        tf.keras.layers.Dense(256, activation='relu'),  # hidden layer
        tf.keras.layers.Dense(256, activation='relu'),  # hidden layer
        tf.keras.layers.Dense(256, activation='relu'),  # hidden layer
        tf.keras.layers.Dense(256, activation='relu'),  # hidden layer
        tf.keras.layers.Dense(1)  # Output layer
    ])
    # Compile model
    nnmodel.compile(loss='mean_absolute_error',
                     optimizer='adam',
                     metrics=['mae'])
    return nnmodel

# Build the model
final_model = create_baseline()

# Train on the full dataset
final_model.fit(X, y,
                epochs=200,
                batch_size=64,
                verbose=1)

Epoch 1/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 13239.0391 - mae: 13239.0391
Epoch 2/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 11037.4277 - mae: 11037.4277
Epoch 3/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 7737.9463 - mae: 7737.9463  
Epoch 4/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 6309.3862 - mae: 6309.3862
Epoch 5/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 4905.5449 - mae: 4905.5449
Epoch 6/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3898.9026 - mae: 3898.9026
Epoch 7/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3546.2654 - mae: 3546.2654
Epoch 8/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3337.7090 - mae: 3337.7090
Epoch 9/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3213.4861 - mae: 3213.4861
Epoch 10/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3280.3882 - mae: 3280.3882
Epoch 11/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3201.4675 - mae: 3201.4675
Epoch 12/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms

In [10]:
#Prediction of new data

#read in the data
new_patients = pd.read_csv('new_patients.csv')

#prepare the data
X_new = pd.get_dummies(new_patients,drop_first=True)

#Make sure it has the same columns as the data it was trained on
columns = X.columns
X_new = X_new.reindex(columns=columns, fill_value=0)

#Make sure it is transformed in the same way as the training data
X_new = scaler.transform(X_new) 

#make the predictions
predictions = final_model.predict(X_new)



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step


In [12]:
#print the predictions
print(predictions)

[[14274.481 ]
 [ 7697.4604]
 [ 5843.768 ]
 [13951.565 ]
 [12270.454 ]
 [ 5517.7046]
 [12295.322 ]
 [ 4922.056 ]
 [ 6573.6724]
 [ 2602.2698]]


In [13]:
#print the predictions in a cleaner table

predictions = predictions.flatten()

results_table = pd.DataFrame({
    'Patient': [f'Patient {i+1}' for i in range(len(predictions))],
    'Prediction': predictions
})

print(results_table)

      Patient    Prediction
0   Patient 1  14274.481445
1   Patient 2   7697.460449
2   Patient 3   5843.768066
3   Patient 4  13951.565430
4   Patient 5  12270.454102
5   Patient 6   5517.704590
6   Patient 7  12295.322266
7   Patient 8   4922.056152
8   Patient 9   6573.672363
9  Patient 10   2602.269775
